In [1]:
# https://www.zhihu.com/question/24533374/answer/127530955922
# https://www.kaggle.com/competitions/classify-leaves/code
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, random_split
import torch.nn.functional as F
from torchvision import datasets, transforms
from sympy.physics.units import degree
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from PIL import Image

base_path = r'E:\AIdata\kaggle-classify-leaves'
train_path = os.path.join(base_path, 'train.csv')
test_path = os.path.join(base_path, 'test.csv')
train_img_path = os.path.join(base_path, 'images')
sample_submission_path = os.path.join(base_path, 'sample_submission.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)
print(f"训练数据预览：\n{train_df.head(10)}")
print(f"测试数据预览：\n{test_df.head(10)}")
print(f"提交样例预览：\n{sample_submission_df.head(10)}")

train_labels, unique_labels = pd.factorize(train_df['label'])
print(train_labels)
print(unique_labels)

训练数据预览：
          image                     label
0  images/0.jpg          maclura_pomifera
1  images/1.jpg          maclura_pomifera
2  images/2.jpg          maclura_pomifera
3  images/3.jpg          maclura_pomifera
4  images/4.jpg          maclura_pomifera
5  images/5.jpg          maclura_pomifera
6  images/6.jpg               ulmus_rubra
7  images/7.jpg  broussonettia_papyrifera
8  images/8.jpg          maclura_pomifera
9  images/9.jpg  broussonettia_papyrifera
测试数据预览：
              image
0  images/18353.jpg
1  images/18354.jpg
2  images/18355.jpg
3  images/18356.jpg
4  images/18357.jpg
5  images/18358.jpg
6  images/18359.jpg
7  images/18360.jpg
8  images/18361.jpg
9  images/18362.jpg
提交样例预览：
              image                   label
0  images/18353.jpg      halesia_tetraptera
1  images/18354.jpg   robinia_pseudo-acacia
2  images/18355.jpg  chionanthus_virginicus
3  images/18356.jpg         ulmus_americana
4  images/18357.jpg           picea_pungens
5  images/18358.jpg        tsu

In [3]:
def get_image_size(img_path: str) -> tuple:
    with Image.open(img_path) as img:
        return img.size  # (width, height)


for relative_path in train_df['image'].sample(n=20):
    image_path = os.path.join(base_path, relative_path)
    resolution = get_image_size(image_path)
    print(f"图片{relative_path} (宽度, 高度)={resolution}")


图片images/10681.jpg (宽度, 高度)=(224, 224)
图片images/9772.jpg (宽度, 高度)=(224, 224)
图片images/12303.jpg (宽度, 高度)=(224, 224)
图片images/12833.jpg (宽度, 高度)=(224, 224)
图片images/302.jpg (宽度, 高度)=(224, 224)
图片images/8136.jpg (宽度, 高度)=(224, 224)
图片images/11758.jpg (宽度, 高度)=(224, 224)
图片images/7788.jpg (宽度, 高度)=(224, 224)
图片images/16186.jpg (宽度, 高度)=(224, 224)
图片images/2711.jpg (宽度, 高度)=(224, 224)
图片images/4380.jpg (宽度, 高度)=(224, 224)
图片images/16915.jpg (宽度, 高度)=(224, 224)
图片images/8122.jpg (宽度, 高度)=(224, 224)
图片images/9641.jpg (宽度, 高度)=(224, 224)
图片images/16741.jpg (宽度, 高度)=(224, 224)
图片images/3454.jpg (宽度, 高度)=(224, 224)
图片images/10457.jpg (宽度, 高度)=(224, 224)
图片images/12766.jpg (宽度, 高度)=(224, 224)
图片images/9143.jpg (宽度, 高度)=(224, 224)
图片images/12798.jpg (宽度, 高度)=(224, 224)


In [24]:
train_df['image'].values

array(['images/0.jpg', 'images/1.jpg', 'images/2.jpg', ...,
       'images/18350.jpg', 'images/18351.jpg', 'images/18352.jpg'],
      dtype=object)

In [2]:
class LeavesDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.image_paths = [os.path.join(root_dir, relative_path) for relative_path in df['image']]
        self.transform = transform
        # 如果 'label' 列不存在则为测试集
        if 'label' in df.columns:
            labels, self.unique_labels = pd.factorize(df['label'])
            self.labels = torch.tensor(labels, dtype=torch.long)
            self.num_classes = len(self.unique_labels)
        else:
            self.labels = None

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        """
        Returns: (image, label) 如果是训练集，image 如果是测试集
        """
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)

        if self.labels is None:  # 测试集
            return image
        else:
            return image, self.labels[idx]


tsf = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # 随机水平翻转
    transforms.RandomRotation(degrees=20),  # 随机旋转
    transforms.ToTensor()
])

train_set = LeavesDataset(train_df, base_path, transform=tsf)
test_set = LeavesDataset(test_df, base_path, transform=tsf)

# 分训练集为 9/10 的训练集和 1/10 的验证集
train_size = int(0.9 * len(train_set))
val_size = len(train_set) - train_size
print(f"训练集大小={train_size}, 验证集大小={val_size}")

# 随机划分数据集
train_subset, val_subset = random_split(train_set, [train_size, val_size])

# num_workers 指定了用于加载数据的子进程数量。每个子进程负责从数据集中加载一部分数据并将其返回给主进程。
# pin_memory 是一个布尔值（True 或 False），用于指定是否使用固定内存（pinned memory）来加速数据传输。
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

训练集大小=16517, 验证集大小=1836


In [8]:
# 检查前10个样本
for i in range(10):
    x, y = train_set[i]
    print(x.shape, y)

torch.Size([3, 224, 224]) tensor(0)
torch.Size([3, 224, 224]) tensor(0)
torch.Size([3, 224, 224]) tensor(0)
torch.Size([3, 224, 224]) tensor(0)
torch.Size([3, 224, 224]) tensor(0)
torch.Size([3, 224, 224]) tensor(0)
torch.Size([3, 224, 224]) tensor(1)
torch.Size([3, 224, 224]) tensor(2)
torch.Size([3, 224, 224]) tensor(0)
torch.Size([3, 224, 224]) tensor(2)


In [17]:
# 检查数据加载器
for batch_idx, (x, y) in enumerate(val_loader):
    print(f'batch {batch_idx}: x.shape={x.shape}, y.shape={y.shape}')

batch 0: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 1: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 2: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 3: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 4: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 5: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 6: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 7: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 8: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 9: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 10: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 11: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 12: x.shape=torch.Size([64, 3, 224, 224]), y.shape=torch.Size([64])
batch 13: x.shape=torch.Size([64, 3, 224, 224]),

In [3]:
class ResNet(nn.Module):
    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1, stride=strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size=1, stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3:
            x = self.conv3(x)
        y += x  # 残差
        return self.relu(y)


b1 = nn.Sequential(nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
                   nn.BatchNorm2d(64),
                   nn.ReLU(),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))


def resnet_block(input_channels, num_channels, num_residuals, first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(ResNet(input_channels, num_channels, use_1x1conv=True, strides=2))
        else:
            blk.append(ResNet(num_channels, num_channels))
    return blk


b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block=True))
b3 = nn.Sequential(*resnet_block(64, 128, 2))
b4 = nn.Sequential(*resnet_block(128, 256, 2))
b5 = nn.Sequential(*resnet_block(256, 512, 2))
net = nn.Sequential(b1, b2, b3, b4, b5,
                    nn.AdaptiveAvgPool2d((1, 1)),
                    nn.Flatten(),
                    nn.Linear(512, train_set.num_classes))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
net.to(device)

cuda


Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (1): Sequential(
    (0): ResNet(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (1): ResNet(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (

In [4]:
def evaluate(model, dataloader, criterion, device='cpu'):
    model.eval()  # 关闭 dropout 和 batch norm 的训练行为
    loss_sum: float = 0.0
    correct_sum: float = 0.0
    num: int = 0
    with torch.no_grad():
        model.to(device)
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss_sum += criterion(logits, y).item()
            y_hat = logits.argmax(dim=1)
            correct_sum += (y_hat == y).sum().item()
            num += y.size(0)
    epoch_loss = loss_sum / num
    epoch_accuracy = correct_sum / num * 100  # 百分比形式
    return epoch_loss, epoch_accuracy


vali_loss, vali_accuracy = evaluate(net, val_loader, nn.CrossEntropyLoss(), device='cuda')
print(vali_loss, vali_accuracy)

0.08166244970167903 0.5991285403050108


In [7]:
def train(model, train_loader, valid_loader,
          num_epochs, learning_rate=0.001, weight_decay=0.01,
          load=False, device='cpu'):
    model.to(device)
    print(f"Model is on device: {next(model.parameters()).device}")
    best_val_accuracy: float = 0.0
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    if load:
        checkpoint = torch.load('../outputs/checkpoints/kaggle_leaves.pth.tar', map_location=device, weights_only=True)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        best_val_accuracy = checkpoint['best_val_accuracy']
        print(f"Model loaded with Val Accuracy: {best_val_accuracy:.2f}%")
    for epoch in range(num_epochs):
        model.train()
        loss_sum: float = 0.0
        correct_sum: float = 0.0
        num: int = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()  # 更新参数
            loss_sum += loss.item()
            y_hat = logits.argmax(dim=1)
            correct_sum += (y_hat == y).sum().item()
            num += y.size(0)
        # 计算当前 epoch 训练集的平均损失和准确率
        train_epoch_loss = loss_sum / num
        train_epoch_accuracy = correct_sum / num * 100

        # 评估训练过的模型在验证集的表现
        vali_loss, vali_accuracy = evaluate(model, valid_loader, criterion, device)
        print(
            f"Epoch [{epoch + 1}/{num_epochs}], train loss {train_epoch_loss:.4f}, train accuracy {train_epoch_accuracy:.2f}%, "
            f"validation loss {vali_loss:.4f}, validation accuracy {vali_accuracy:.2f}%")
        print('=' * 50)

        # 保存检查点
        if vali_accuracy > best_val_accuracy:
            best_val_accuracy = vali_accuracy
            checkpoint = {'model_state_dict': model.state_dict(),
                          'optimizer_state_dict': optimizer.state_dict(),
                          'best_val_accuracy': best_val_accuracy}
            torch.save(checkpoint, f'../outputs/checkpoints/kaggle_leaves.pth.tar', _use_new_zipfile_serialization=True)
            print(f"Checkpoint saved at epoch {epoch + 1} with Val Accuracy: {vali_accuracy:.2f}%")


train(net, train_loader, val_loader, 5, 0.00001, 0.01, True, device)

Model is on device: cuda:0
Model loaded with Val Accuracy: 98.42%
Epoch [1/5], train loss 0.0010, train accuracy 98.23%, validation loss 0.0011, validation accuracy 98.15%
Epoch [2/5], train loss 0.0010, train accuracy 98.24%, validation loss 0.0011, validation accuracy 98.09%
Epoch [3/5], train loss 0.0009, train accuracy 98.35%, validation loss 0.0011, validation accuracy 97.66%
Epoch [4/5], train loss 0.0009, train accuracy 98.49%, validation loss 0.0011, validation accuracy 97.93%
Epoch [5/5], train loss 0.0009, train accuracy 98.38%, validation loss 0.0012, validation accuracy 97.93%


In [ ]:
def save_checkpoint(state, save_dir='', filename='checkpoint.pth.tar'):
    ckpt = {'model_state_dict': state['model'].state_dict(),
            'optimizer_state_dict': state['optimizer'].state_dict()}
    torch.save(state, filename)

In [ ]:
def load_checkpoint(checkpoint):
    print(f">> Loading checkpoint...")
    model.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])